In [ ]:
import pandas as pd

df = pd.read_csv("/content/creditcard.csv")

print(df.shape)
print("Missing values:", df.isnull().sum().sum())
print(df["Class"].value_counts())

In [ ]:
fraud_rate = df["Class"].mean()

print(f"Fraud rate: {fraud_rate:.4%}")
print(f"Normal transactions: {(df['Class'] == 0).sum():,}")
print(f"Fraud transactions: {(df['Class'] == 1).sum():,}")

In [ ]:
df.groupby("Class")["Amount"].describe()

In [ ]:
fraud_rate = df["Class"].mean()

print(f"Fraud rate: {fraud_rate:.4%}")
print(f"Normal transactions: {(df['Class'] == 0).sum():,}")
print(f"Fraud transactions: {(df['Class'] == 1).sum():,}")

In [ ]:
df.groupby("Class")["Amount"].describe()

In [ ]:
normal = df[df["Class"] == 0]
fraud = df[df["Class"] == 1]

print("Normal transaction amount:")
print(normal["Amount"].describe())

print("\nFraud transaction amount:")
print(fraud["Amount"].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(
    normal["Amount"],
    bins=50,
    alpha=0.6,
    label="Normal"
)

plt.hist(
    fraud["Amount"],
    bins=50,
    alpha=0.6,
    label="Fraud"
)

plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.title("Transaction Amount Distribution")
plt.legend()

plt.show()

In [ ]:
df.groupby("Class")["Time"].describe()

In [ ]:
df["Hour"] = (df["Time"] / 3600) % 24

In [ ]:
df[["Time", "Hour", "Amount", "Class"]].head()

In [ ]:
df["Hour"] = ((df["Time"] / 3600) % 24).astype(int)

normal = df[df["Class"] == 0].copy()
fraud = df[df["Class"] == 1].copy()

fraud.groupby("Hour").size()

In [ ]:
fraud.groupby("Hour").size().plot(
    kind="bar",
    figsize=(10, 5),
    title="Fraud Transactions by Hour"
)

In [ ]:
hourly_stats = df.groupby("Hour").agg(
    total_transactions=("Class", "count"),
    fraud_transactions=("Class", "sum")
)

hourly_stats["fraud_rate"] = (
    hourly_stats["fraud_transactions"]
    / hourly_stats["total_transactions"]
)

hourly_stats

In [ ]:
import matplotlib.pyplot as plt

hourly_stats["fraud_rate"].plot(
    kind="bar",
    figsize=(10, 5),
    title="Fraud Rate by Relative Hour"
)

plt.xlabel("Relative Hour")
plt.ylabel("Fraud Rate")
plt.show()

In [ ]:
df.groupby("Class")["Amount"].describe()

In [ ]:
print("Normal transactions:")
print(df[df["Class"] == 0]["Amount"].describe())

print("\nFraud transactions:")
print(df[df["Class"] == 1]["Amount"].describe())

In [ ]:
X = df.drop(columns=["Class", "Hour"])
y = df["Class"]

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining fraud:")
print(y_train.value_counts())

print("\nTest fraud:")
print(y_test.value_counts())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

In [ ]:
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print(classification_report(y_test, y_pred, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

balanced_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

balanced_model.fit(X_train, y_train)

In [ ]:
y_pred_balanced = balanced_model.predict(X_test)
y_prob_balanced = balanced_model.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(
    y_test,
    y_pred_balanced,
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_test,
    y_pred_balanced
))

print(
    "ROC-AUC:",
    roc_auc_score(y_test, y_prob_balanced)
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        y_prob_balanced
    )
)

In [ ]:
y_prob

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for threshold in thresholds:

    y_pred_threshold = (
        y_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold
    )

    recall = recall_score(
        y_test,
        y_pred_threshold
    )

    f1 = f1_score(
        y_test,
        y_pred_threshold
    )

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

FN_COST = 500   # hypothetical loss per missed fraud
FP_COST = 5     # hypothetical cost per false positive

results = []

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred_threshold
    ).ravel()

    total_cost = fn * FN_COST + fp * FP_COST

    results.append({
        "threshold": threshold,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "cost": total_cost
    })

cost_df = pd.DataFrame(results)

cost_df

In [ ]:
cost_df.sort_values("cost")

In [ ]:
import numpy as np

thresholds = np.arange(0.01, 0.51, 0.01)

results = []

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred_threshold
    ).ravel()

    total_cost = fn * FN_COST + fp * FP_COST

    results.append({
        "threshold": threshold,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "cost": total_cost
    })

cost_df = pd.DataFrame(results)

cost_df.sort_values("cost").head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    cost_df["threshold"],
    cost_df["cost"]
)

plt.xlabel("Decision Threshold")
plt.ylabel("Expected Cost")
plt.title("Expected Risk Cost by Decision Threshold")

plt.show()

In [ ]:
cost_df.sort_values("cost").head(10)

In [ ]:
best_row = cost_df.loc[cost_df["cost"].idxmin()]

print("Best threshold:", best_row["threshold"])
print("True Positives:", int(best_row["TP"]))
print("False Negatives:", int(best_row["FN"]))
print("False Positives:", int(best_row["FP"]))
print("True Negatives:", int(best_row["TN"]))
print("Minimum simulated cost:", best_row["cost"])

In [ ]:
best_threshold = best_row["threshold"]

y_pred_best = (y_prob >= best_threshold).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score

print(
    "Precision:",
    precision_score(y_test, y_pred_best)
)

print(
    "Recall:",
    recall_score(y_test, y_pred_best)
)

print(
    "F1:",
    f1_score(y_test, y_pred_best)
)

In [ ]:
cost_df.sort_values("cost").head(10)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class", "Hour"])
y = df["Class"]

# 先拿出 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 剩下的 80% 再拆成 60% train + 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nFraud counts:")
print("Train:", y_train.sum())
print("Validation:", y_val.sum())
print("Test:", y_test.sum())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

In [ ]:
y_prob_val = baseline_model.predict_proba(X_val)[:, 1]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

FN_COST = 500
FP_COST = 5

thresholds = np.arange(0.01, 0.51, 0.01)

results = []

for threshold in thresholds:

    y_pred_val = (y_prob_val >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred_val
    ).ravel()

    cost = fn * FN_COST + fp * FP_COST

    results.append({
        "threshold": threshold,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "cost": cost
    })

val_cost_df = pd.DataFrame(results)

val_cost_df.sort_values("cost").head(10)

In [ ]:
best_threshold = val_cost_df.loc[
    val_cost_df["cost"].idxmin(),
    "threshold"
]

print("Best validation threshold:", best_threshold)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

y_prob_test = baseline_model.predict_proba(X_test)[:, 1]

y_pred_test = (
    y_prob_test >= best_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred_test
).ravel()

test_cost = fn * FN_COST + fp * FP_COST

print("Threshold:", best_threshold)
print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

print("Precision:", precision_score(y_test, y_pred_test))
print("Recall:", recall_score(y_test, y_pred_test))
print("F1:", f1_score(y_test, y_pred_test))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_test))
print("PR-AUC:", average_precision_score(y_test, y_prob_test))

print("Simulated test cost:", test_cost)

In [ ]:
from xgboost import XGBClassifier

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Normal:", negative_count)
print("Fraud:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

xgb_prob_val = xgb_model.predict_proba(X_val)[:, 1]

print(
    "Validation ROC-AUC:",
    roc_auc_score(y_val, xgb_prob_val)
)

print(
    "Validation PR-AUC:",
    average_precision_score(y_val, xgb_prob_val)
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

FN_COST = 500
FP_COST = 5

thresholds = np.arange(0.01, 0.51, 0.01)

xgb_results = []

for threshold in thresholds:

    y_pred_val_xgb = (
        xgb_prob_val >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred_val_xgb
    ).ravel()

    cost = fn * FN_COST + fp * FP_COST

    xgb_results.append({
        "threshold": threshold,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "cost": cost
    })

xgb_cost_df = pd.DataFrame(xgb_results)

xgb_cost_df.sort_values("cost").head(10)

In [ ]:
best_xgb_threshold = xgb_cost_df.loc[
    xgb_cost_df["cost"].idxmin(),
    "threshold"
]

print("Best XGBoost validation threshold:", best_xgb_threshold)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

xgb_prob_test = xgb_model.predict_proba(X_test)[:, 1]

xgb_pred_test = (
    xgb_prob_test >= best_xgb_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    xgb_pred_test
).ravel()

xgb_test_cost = (
    fn * FN_COST
    + fp * FP_COST
)

print("Threshold:", best_xgb_threshold)

print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

print(
    "Precision:",
    precision_score(y_test, xgb_pred_test)
)

print(
    "Recall:",
    recall_score(y_test, xgb_pred_test)
)

print(
    "F1:",
    f1_score(y_test, xgb_pred_test)
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test, xgb_prob_test)
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        xgb_prob_test
    )
)

print(
    "Simulated test cost:",
    xgb_test_cost
)

In [ ]:
xgb_unweighted = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

xgb_unweighted.fit(X_train, y_train)

In [ ]:
xgb_unweighted_prob_val = xgb_unweighted.predict_proba(X_val)[:, 1]

print(
    "Validation ROC-AUC:",
    roc_auc_score(y_val, xgb_unweighted_prob_val)
)

print(
    "Validation PR-AUC:",
    average_precision_score(
        y_val,
        xgb_unweighted_prob_val
    )
)

In [ ]:
thresholds = np.arange(0.001, 0.501, 0.001)

In [ ]:
results = []

for threshold in thresholds:

    pred = (
        xgb_unweighted_prob_val >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        pred
    ).ravel()

    cost = fn * FN_COST + fp * FP_COST

    results.append({
        "threshold": threshold,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "cost": cost
    })

xgb_unweighted_cost_df = pd.DataFrame(results)

xgb_unweighted_cost_df.sort_values("cost").head(10)

In [ ]:
best_xgb_unweighted_threshold = (
    xgb_unweighted_cost_df.loc[
        xgb_unweighted_cost_df["cost"].idxmin(),
        "threshold"
    ]
)

print(
    "Best validation threshold:",
    best_xgb_unweighted_threshold
)

In [ ]:
xgb_unweighted_prob_test = (
    xgb_unweighted.predict_proba(X_test)[:, 1]
)

xgb_unweighted_pred_test = (
    xgb_unweighted_prob_test
    >= best_xgb_unweighted_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    xgb_unweighted_pred_test
).ravel()

cost = fn * FN_COST + fp * FP_COST

print("Threshold:", best_xgb_unweighted_threshold)
print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

print(
    "Precision:",
    precision_score(
        y_test,
        xgb_unweighted_pred_test
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        xgb_unweighted_pred_test
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        xgb_unweighted_pred_test
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        xgb_unweighted_prob_test
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        xgb_unweighted_prob_test
    )
)

print("Simulated test cost:", cost)

In [ ]:
import shap

In [ ]:
X_shap = X_test.sample(
    n=5000,
    random_state=42
)

In [ ]:
explainer = shap.TreeExplainer(xgb_unweighted)

shap_values = explainer(X_shap)

In [ ]:
shap.plots.bar(
    shap_values,
    max_display=15
)

In [ ]:
shap.plots.beeswarm(
    shap_values,
    max_display=15
)

In [ ]:
test_results = X_test.copy()

test_results["actual"] = y_test
test_results["fraud_score"] = xgb_unweighted_prob_test
test_results["prediction"] = xgb_unweighted_pred_test

In [ ]:
def get_error_type(row):

    if row["actual"] == 1 and row["prediction"] == 0:
        return "False Negative"

    elif row["actual"] == 0 and row["prediction"] == 1:
        return "False Positive"

    elif row["actual"] == 1 and row["prediction"] == 1:
        return "True Positive"

    else:
        return "True Negative"


test_results["error_type"] = test_results.apply(
    get_error_type,
    axis=1
)

In [ ]:
test_results["error_type"].value_counts()

In [ ]:
false_negatives = test_results[
    test_results["error_type"] == "False Negative"
]

false_negatives[
    ["Amount", "Time", "fraud_score"]
].sort_values(
    "fraud_score",
    ascending=False
)

In [ ]:
false_positives = test_results[
    test_results["error_type"] == "False Positive"
]

false_positives[
    ["Amount", "Time", "fraud_score"]
].sort_values(
    "fraud_score",
    ascending=False
).head(20)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Weighted XGBoost",
        "Unweighted XGBoost"
    ],

    "Threshold": [
        0.010,
        0.050,
        0.006
    ],

    "Precision": [
        0.36596,
        0.17886,
        0.31769
    ],

    "Recall": [
        0.87755,
        0.89796,
        0.89796
    ],

    "PR-AUC": [
        0.74485,
        0.86065,
        0.87640
    ],

    "False Negatives": [
        12,
        10,
        10
    ],

    "False Positives": [
        149,
        404,
        189
    ],

    "Simulated Cost": [
        6745,
        7020,
        5945
    ]
})

comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Weighted XGBoost",
        "Unweighted XGBoost"
    ],

    "Threshold": [
        0.010,
        0.050,
        0.006
    ],

    "Precision": [
        0.36596,
        0.17886,
        0.31769
    ],

    "Recall": [
        0.87755,
        0.89796,
        0.89796
    ],

    "PR-AUC": [
        0.74485,
        0.86065,
        0.87640
    ],

    "False Negatives": [
        12,
        10,
        10
    ],

    "False Positives": [
        149,
        404,
        189
    ],

    "Simulated Cost": [
        6745,
        7020,
        5945
    ]
})

comparison

In [ ]:
import shap

In [ ]:
X_shap = X_test.sample(
    n=5000,
    random_state=42
)

print(X_shap.shape)

In [ ]:
explainer = shap.TreeExplainer(xgb_unweighted)

shap_values = explainer(X_shap)

In [ ]:
shap.plots.bar(
    shap_values,
    max_display=15
)

In [ ]:
shap.plots.beeswarm(
    shap_values,
    max_display=15
)

In [ ]:
test_results = X_test.copy()

test_results["actual"] = y_test
test_results["fraud_score"] = xgb_unweighted_prob_test
test_results["prediction"] = xgb_unweighted_pred_test

In [ ]:
def get_error_type(row):
    if row["actual"] == 1 and row["prediction"] == 0:
        return "False Negative"
    elif row["actual"] == 0 and row["prediction"] == 1:
        return "False Positive"
    elif row["actual"] == 1 and row["prediction"] == 1:
        return "True Positive"
    else:
        return "True Negative"

test_results["error_type"] = test_results.apply(
    get_error_type,
    axis=1
)

test_results["error_type"].value_counts()

In [ ]:
false_negatives = test_results[
    test_results["error_type"] == "False Negative"
]

false_negatives[
    ["Amount", "Time", "fraud_score"]
].sort_values(
    "fraud_score",
    ascending=False
)

In [ ]:
false_negatives["fraud_score"].describe()

In [ ]:
false_positives = test_results[
    test_results["error_type"] == "False Positive"
]

false_positives[
    ["Amount", "Time", "fraud_score"]
].sort_values(
    "fraud_score",
    ascending=False
).head(20)

In [ ]:
# Highest-confidence False Positive
fp_case = (
    test_results[
        test_results["error_type"] == "False Positive"
    ]
    .sort_values("fraud_score", ascending=False)
    .iloc[0]
)

# Lowest-score False Negative
fn_case = (
    test_results[
        test_results["error_type"] == "False Negative"
    ]
    .sort_values("fraud_score", ascending=True)
    .iloc[0]
)

print("Highest-confidence False Positive")
print("Index:", fp_case.name)
print("Amount:", fp_case["Amount"])
print("Time:", fp_case["Time"])
print("Fraud score:", fp_case["fraud_score"])

print("\nLowest-score False Negative")
print("Index:", fn_case.name)
print("Amount:", fn_case["Amount"])
print("Time:", fn_case["Time"])
print("Fraud score:", fn_case["fraud_score"])

In [ ]:
feature_cols = X.columns.tolist()

fp_index = fp_case.name

fp_features = X_test.loc[[fp_index], feature_cols]

fp_features

In [ ]:
fp_shap = explainer(fp_features)

In [ ]:
shap.plots.waterfall(
    fp_shap[0],
    max_display=15
)

In [ ]:
fn_index = fn_case.name

fn_features = X_test.loc[[fn_index], feature_cols]

fn_features

In [ ]:
fn_shap = explainer(fn_features)

In [ ]:
shap.plots.waterfall(
    fn_shap[0],
    max_display=15
)

In [ ]:
explainer = shap.TreeExplainer(xgb_unweighted)

In [ ]:
xgb_unweighted.predict_proba(fp_features)

In [ ]:
fp_prob = xgb_unweighted.predict_proba(fp_features)[:, 1]

print("FP fraud probability:", fp_prob[0])

In [ ]:
fn_prob = xgb_unweighted.predict_proba(fn_features)[:, 1]

print("FN fraud probability:", fn_prob[0])

In [ ]:
xgb_unweighted.predict_proba(X_test)[:, 1]

In [ ]:
import numpy as np

fp_raw_score = (
    fp_shap.base_values[0]
    + fp_shap.values[0].sum()
)

fp_probability = 1 / (1 + np.exp(-fp_raw_score))

print("SHAP raw score:", fp_raw_score)
print("Converted probability:", fp_probability)

In [ ]:
fn_raw_score = (
    fn_shap.base_values[0]
    + fn_shap.values[0].sum()
)

fn_probability = 1 / (1 + np.exp(-fn_raw_score))

print("SHAP raw score:", fn_raw_score)
print("Converted probability:", fn_probability)